# Depth Map Injection Experiment

For each MindCube question, we run DepthAnything v2 on every RGB image to produce a
grayscale depth map, then append the depth maps as extra input images to LLaVA-OneVision.

**Setup (same as baseline notebook):**
- Drive layout: `MyDrive/MindCube/data/raw/MindCube_tinybench.jsonl` + `MyDrive/MindCube/data/`
- Model at: `MyDrive/models/llava-onevision-qwen2-7b-ov-hf`
- DepthAnything v2 model will be downloaded from HuggingFace (~400 MB, small variant)
- Runtime: **A100 GPU** (required)

Outputs: `/content/depth_injection_results.jsonl`

In [1]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q transformers>=4.45.0 accelerate>=0.27.0 huggingface_hub pillow tqdm

In [2]:
# ── 2. Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH  = "/content/drive/MyDrive/MindCube/data/raw/MindCube_tinybench.jsonl"
IMAGE_ROOT = "/content/drive/MyDrive/MindCube/data/"
LLAVA_ID   = "/content/drive/MyDrive/models/llava-onevision-qwen2-7b-ov-hf"

import pathlib
assert pathlib.Path(DATA_PATH).exists(), f"Not found: {DATA_PATH}"
print("Data found.")

Mounted at /content/drive
Data found.


In [3]:
# ── 2b. Download DepthAnything v2-Small to Drive (skip if already there) ─────
import pathlib
from huggingface_hub import snapshot_download

DA_DRIVE_PATH = "/content/drive/MyDrive/models/depth-anything-v2-small-hf"

if pathlib.Path(DA_DRIVE_PATH).exists():
    print(f"DepthAnything already on Drive at {DA_DRIVE_PATH} — skipping download.")
else:
    print("Downloading DepthAnything v2-Small to Drive (~400 MB)...")
    snapshot_download(
        repo_id="depth-anything/Depth-Anything-V2-Small-hf",
        repo_type="model",
        local_dir=DA_DRIVE_PATH,
    )
    print("Download complete.")

DepthAnything already on Drive at /content/drive/MyDrive/models/depth-anything-v2-small-hf — skipping download.


In [4]:
# ── 3. Load DepthAnything v2 (from Drive) ────────────────────────────────────
import torch
from transformers import pipeline
from PIL import Image
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

depth_pipe = pipeline(
    task="depth-estimation",
    model=DA_DRIVE_PATH,
    device=0 if device == "cuda" else -1,
)
print("DepthAnything v2 loaded.")


def make_depth_image(rgb_image: Image.Image) -> Image.Image:
    """Run DepthAnything v2, return a grayscale PIL Image resized to match input."""
    out = depth_pipe(rgb_image)
    depth_arr = np.array(out["depth"])  # float32 array
    d_min, d_max = depth_arr.min(), depth_arr.max()
    if d_max > d_min:
        depth_u8 = ((depth_arr - d_min) / (d_max - d_min) * 255).astype(np.uint8)
    else:
        depth_u8 = np.zeros_like(depth_arr, dtype=np.uint8)
    gray = Image.fromarray(depth_u8, mode="L").convert("RGB")  # LLaVA expects RGB
    gray = gray.resize(rgb_image.size, Image.BILINEAR)
    return gray


# Quick sanity check
from PIL import Image as _PIL
test_img = _PIL.new("RGB", (224, 224), color=(128, 64, 32))
test_depth = make_depth_image(test_img)
print(f"Depth map size: {test_depth.size}, mode: {test_depth.mode}")

Device: cuda


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


DepthAnything v2 loaded.
Depth map size: (224, 224), mode: RGB


/tmp/ipykernel_9676/3350012156.py:27: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  gray = Image.fromarray(depth_u8, mode="L").convert("RGB")  # LLaVA expects RGB


In [5]:
# ── 4. Load LLaVA-OneVision ──────────────────────────────────────────────────
import gc
from transformers import LlavaOnevisionForConditionalGeneration, AutoProcessor

dtype = torch.float16

print(f"Loading LLaVA from {LLAVA_ID} ...")
processor = AutoProcessor.from_pretrained(LLAVA_ID)
model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    LLAVA_ID,
    torch_dtype=dtype,
    device_map="auto",
    attn_implementation="sdpa",
)
model.eval()

# Disable AnyRes tiling — each image becomes 1 tile (729 tokens)
processor.image_processor.do_image_splitting = False

gc.collect()
torch.cuda.empty_cache()
print(f"LLaVA loaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated.")

Loading LLaVA from /content/drive/MyDrive/models/llava-onevision-qwen2-7b-ov-hf ...


The image processor of type `LlavaOnevisionImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/765 [00:00<?, ?it/s]

LLaVA loaded. GPU memory: 16.2 GB allocated.


In [6]:
# ── 5. Dataset and inference helpers ─────────────────────────────────────────
import json, re
from pathlib import Path
from collections import defaultdict
from torch.utils.data import Dataset
from tqdm.notebook import tqdm

_HEADER = "Look at these images carefully. They show a scene from different viewpoints.\n\n"
_FOOTER = "\n\nAnswer with one letter only (A, B, C, or D)."

_TAG  = re.compile(r"<answer>\s*([A-E])", re.I)
_DECL = re.compile(r"(?:the\s+answer\s+is|answer\s*:)\s*([A-E])\.?", re.I)
_LINE = re.compile(r"^\s*([A-E])[\.):]?\s*$", re.I | re.M)
_ANY  = re.compile(r"([A-E])", re.I)


def extract_answer(text):
    for pat in [_TAG, _DECL]:
        m = pat.search(text)
        if m:
            return m.group(1).upper()
    for pat in [_LINE, _ANY]:
        ms = pat.findall(text)
        if ms:
            return ms[-1].upper()
    return None


class MindCubeDataset(Dataset):
    def __init__(self, jsonl_path, image_root, max_samples=None):
        self.image_root = Path(image_root)
        self.samples = []
        with open(jsonl_path) as f:
            for line in f:
                line = line.strip()
                if line:
                    self.samples.append(json.loads(line))
                    if max_samples and len(self.samples) >= max_samples:
                        break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        record = self.samples[idx]
        images = [Image.open(self.image_root / r).convert("RGB") for r in record["images"]]
        prompt = _HEADER + record["question"] + _FOOTER
        return {
            "id": record["id"],
            "images": images,
            "prompt": prompt,
            "gt_answer": record["gt_answer"],
            "setting": record["id"].split("_")[0].lower(),
            "n_rgb": len(images),
        }


@torch.inference_mode()
def generate_with_depth(rgb_images, depth_images, prompt, max_new_tokens=128):
    """
    Feed LLaVA the RGB images followed by their corresponding depth maps.
    The prompt is updated to tell the model what the extra images are.
    """
    n_rgb = len(rgb_images)
    all_images = rgb_images + depth_images
    depth_note = (
        f" The last {n_rgb} image(s) are grayscale depth maps "
        "(brighter = closer) corresponding to the scene views above."
    )
    full_prompt = prompt.replace(
        "\n\nAnswer with one letter only",
        depth_note + "\n\nAnswer with one letter only",
    )
    content = [{"type": "image"} for _ in all_images] + [{"type": "text", "text": full_prompt}]
    conversation = [{"role": "user", "content": content}]
    text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(images=all_images, text=text, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = out[0][inputs["input_ids"].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True).strip()


print("Helpers ready.")

Helpers ready.


In [7]:
# ── 6. Sanity check (3 samples) ──────────────────────────────────────────────
ds_smoke = MindCubeDataset(DATA_PATH, IMAGE_ROOT, max_samples=3)
for i in range(len(ds_smoke)):
    s = ds_smoke[i]
    depth_imgs = [make_depth_image(img) for img in s["images"]]
    raw = generate_with_depth(s["images"], depth_imgs, s["prompt"])
    pred = extract_answer(raw)
    print(f"[{s['id']}]  gt={s['gt_answer']}  pred={pred}  raw={repr(raw[:80])}")
    for img in s["images"] + depth_imgs:
        img.close()

/tmp/ipykernel_9676/3350012156.py:27: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  gray = Image.fromarray(depth_u8, mode="L").convert("RGB")  # LLaVA expects RGB
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[among_group693_q1_5_2]  gt=C  pred=D  raw='D'


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[among_group458_q0_2_3]  gt=A  pred=A  raw='A'


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[among_group603_q1_1_2]  gt=B  pred=C  raw='C'


In [8]:
# ── 7. Full evaluation (all 1050 samples) ────────────────────────────────────
MAX_SAMPLES = None  # None = all 1050
OUT_PATH = "/content/depth_injection_results.jsonl"

dataset = MindCubeDataset(DATA_PATH, IMAGE_ROOT, MAX_SAMPLES)
print(f"{len(dataset)} samples")

results = []
with open(OUT_PATH, "w") as f_out:
    for idx in tqdm(range(len(dataset))):
        sample = dataset[idx]
        depth_imgs = []
        try:
            depth_imgs = [make_depth_image(img) for img in sample["images"]]
            raw = generate_with_depth(sample["images"], depth_imgs, sample["prompt"])
            predicted = extract_answer(raw)
            error = None
        except Exception as e:
            raw, predicted, error = "", None, str(e)
            tqdm.write(f"[WARN] {sample['id']}: {str(e)[:120]}")

        for img in sample["images"] + depth_imgs:
            img.close()

        gt = (sample["gt_answer"] or "").upper()
        result = {
            "id": sample["id"],
            "setting": sample["setting"],
            "gt_answer": gt,
            "predicted": predicted,
            "correct": predicted is not None and predicted == gt,
            "raw_output": raw,
            **(({"error": error}) if error else {}),
        }
        results.append(result)
        f_out.write(json.dumps(result) + "\n")

        if idx % 50 == 0:
            torch.cuda.empty_cache()
            gc.collect()

print(f"Done. Saved to {OUT_PATH}")

1050 samples


  0%|          | 0/1050 [00:00<?, ?it/s]

/tmp/ipykernel_9676/3350012156.py:27: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  gray = Image.fromarray(depth_u8, mode="L").convert("RGB")  # LLaVA expects RGB
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for op

Done. Saved to /content/depth_injection_results.jsonl


In [9]:
# ── 8. Metrics ───────────────────────────────────────────────────────────────
BASELINE = {"around": 0.652, "among": 0.418, "rotation": 0.345, "overall": 0.460}

overall = {"correct": 0, "total": 0}
by_setting = defaultdict(lambda: {"correct": 0, "total": 0})

for r in results:
    ok = r["predicted"] is not None and r["predicted"] == r["gt_answer"]
    overall["total"] += 1
    overall["correct"] += int(ok)
    s = r["setting"]
    by_setting[s]["total"] += 1
    by_setting[s]["correct"] += int(ok)

acc = lambda d: d["correct"] / d["total"] if d["total"] else 0.0

print(f"\n{'='*54}")
print(f"  {'Setting':<18}  {'Baseline':>8}  {'DepthInj':>8}  {'Delta':>6}")
print(f"  {'-'*50}")
for s, m in sorted(by_setting.items()):
    a = acc(m)
    b = BASELINE.get(s, 0.0)
    print(f"  {s:<18}  {b:>8.3f}  {a:>8.3f}  {a-b:>+6.3f}")
ov = acc(overall)
b_ov = BASELINE["overall"]
print(f"  {'Overall':<18}  {b_ov:>8.3f}  {ov:>8.3f}  {ov-b_ov:>+6.3f}")
print(f"{'='*54}")

unanswered = sum(1 for r in results if r["predicted"] is None)
errors = sum(1 for r in results if "error" in r)
print(f"Unanswered: {unanswered}/{len(results)}  (errors: {errors})")


  Setting             Baseline  DepthInj   Delta
  --------------------------------------------------
  among                  0.418     0.427  +0.009
  around                 0.652     0.644  -0.008
  rotation               0.345     0.350  +0.005
  Overall                0.460     0.464  +0.004
Unanswered: 0/1050  (errors: 0)


In [10]:
# ── 9. Download results ───────────────────────────────────────────────────────
from google.colab import files
files.download(OUT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>